In [0]:
import requests
import pandas as pd
import os
import json
import numpy as np
import glob
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType,StructType, StructField, StringType, IntegerType, DoubleType,TimestampType,LongType
from delta.tables import DeltaTable
from pyspark.sql.window import Window
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

In [0]:
API_KEY = dbutils.secrets.get(scope="my-scope", key="API_KEY")  # change scope, key as per what was set
SENDER_EMAIL = dbutils.secrets.get(scope='email-scope', key='sender-email')  # change scope, key as per what was set
SENDER_PASSWORD = dbutils.secrets.get(scope='email-scope', key='sender-password').replace(' ', '')  # change scope, key as per what was set
RECEIVER_EMAIL = dbutils.secrets.get(scope='email-scope', key='receiver-email')  # change scope, key as per what was set
FILE_PATH = dbutils.secrets.get(scope="my-scope", key="FILE_PATH")  # change path as per need

In [0]:
SERIES_ID=['FEDFUNDS', 'CPIAUCSL', 'MORTGAGE30US', 'UNRATE','GDP','DGS10','M2SL','USREC']

In [0]:
def read_from_api(URL,file_path,folder,filename):
    full_path=f'{file_path}/{folder}'
    os.makedirs(full_path,exist_ok=True)
    path=f'{full_path}/{filename}.json'
    req=requests.get(URL)
    if req.status_code==200:
        with open(path,'wb') as f:
            for chunk in req.iter_content(chunk_size=1024):
                f.write(chunk)
for s in SERIES_ID:
    INDICATORS_URL=f'https://api.stlouisfed.org/fred/series?series_id={s}&api_key={API_KEY}&file_type=json'
    OBSERVATION_URL=f'https://api.stlouisfed.org/fred/series/observations?series_id={s}&api_key={API_KEY}&file_type=json'
    read_from_api(INDICATORS_URL,FILE_PATH,'indicators',f'indicators_{s}')
    read_from_api(OBSERVATION_URL,FILE_PATH,'observation',f'observation_{s}')

In [0]:
def read_json_to_df(file_path,column=None):
    json_files=glob.glob(os.path.join(file_path,'*.json'))
    dfs=[]
    for file in json_files:
        with open(file,'r') as f:
            series_id = os.path.basename(file).replace('.json', '').split('_', 1)[1]
            data=json.load(f)
            if column in data:
                df=pd.json_normalize(data[column])
                df['series_id'] = series_id
                dfs.append(df)
            else:
                print(f"{column} not found in {file}")
                dfs.append(pd.json_normalize(data))
    return pd.concat(dfs,ignore_index=True)

In [0]:
obs_df=spark.createDataFrame(read_json_to_df(f'{FILE_PATH}/observation','observations'))

In [0]:
obs_df=obs_df.withColumn('value',F.when(F.col('value')=='.',None).otherwise(F.col('value')))

In [0]:
cols_to_drop=['realtime_start','realtime_end']
obs_df=obs_df.withColumn('date',F.to_date(obs_df['date'])).withColumn('value',F.col('value').cast(DecimalType(10, 2))).drop(*cols_to_drop).withColumnRenamed('date','obs_date')

In [0]:
ind_df=read_json_to_df(f'{FILE_PATH}/indicators','seriess')
ind_df=spark.createDataFrame(ind_df)

In [0]:
ind_df=ind_df.withColumnRenamed('title','series_name').withColumnRenamed('notes','description')

In [0]:
cols_to_keep=['id','series_name','description','frequency','units']
ind_df=ind_df.select(*cols_to_keep).withColumnRenamed('id','series_id')

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS us_macroeconomics_tracker.bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS us_macroeconomics_tracker.bronze.indicators (
  `series_id` STRING NOT NULL,
  `series_name` STRING NOT NULL,
  `description` STRING,
  `frequency` STRING NOT NULL,
  `units` STRING NOT NULL
)
USING DELTA

In [0]:
%sql
CREATE TABLE IF NOT EXISTS us_macroeconomics_tracker.bronze.observations (
  `observation_id` BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  `series_id` STRING NOT NULL,
  `obs_date` DATE NOT NULL,
  `value` DOUBLE
)
USING DELTA

In [0]:
%sql
CREATE TABLE IF NOT EXISTS us_macroeconomics_tracker.bronze.ingestion_log (
  `log_id` BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  `table` STRING NOT NULL,
  `series_id` STRING NOT NULL,
  `error_message` STRING,
  `pulled_at` TIMESTAMP NOT NULL,
  `rows_inserted` BIGINT NOT NULL,
  `status` STRING NOT NULL
)  
USING DELTA

In [0]:
%sql
CREATE TABLE IF NOT EXISTS us_macroeconomics_tracker.bronze.dq_log (
  `log_id` BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  `check_name` STRING NOT NULL,
  `table_name` STRING NOT NULL,
  `series_id` STRING,
  `status` STRING NOT NULL,
  `failed_count` BIGINT NOT NULL,
  `run_at` TIMESTAMP NOT NULL
)  
USING DELTA

In [0]:
ing_schema=StructType([
    StructField('table',StringType(),False),
    StructField('series_id',StringType(),False),
    StructField('error_message',StringType(),True),
    StructField('pulled_at',TimestampType(),False),
    StructField('rows_inserted',LongType(),False),
    StructField('status',StringType(),False)
])

In [0]:
for s in SERIES_ID:
    try:
        filtered_ind = ind_df.filter(F.col('series_id') == s)
        ind_row_count=filtered_ind.count()
        DeltaTable.forName(spark, 'us_macroeconomics_tracker.bronze.indicators').merge(filtered_ind.alias('source'), 'us_macroeconomics_tracker.bronze.indicators.series_id=source.series_id').whenNotMatchedInsert(values={'series_id':'source.series_id','series_name':'source.series_name','description':'source.description','frequency':'source.frequency','units':'source.units'}).execute()
        ing_log=spark.createDataFrame([{'table':'indicators','series_id':s,'error_message':None,'pulled_at':datetime.now(),'rows_inserted':ind_row_count,'status':'Success'}],ing_schema).write.format("delta").mode("append").saveAsTable('us_macroeconomics_tracker.bronze.ingestion_log')
    except Exception as e:
        print(f'Error: {e}')
        ing_log=spark.createDataFrame([{'table':'indicators','series_id':s,'error_message':str(e),'pulled_at':datetime.now(),'rows_inserted':0,'status':'Failure'}],ing_schema).write.format("delta").mode("append").saveAsTable('us_macroeconomics_tracker.bronze.ingestion_log')
    try:
        filtered_obs = obs_df.filter(F.col('series_id') == s)
        obs_row_count=filtered_obs.count()
        DeltaTable.forName(spark, 'us_macroeconomics_tracker.bronze.observations').merge(filtered_obs.alias('source'), 'us_macroeconomics_tracker.bronze.observations.series_id=source.series_id and us_macroeconomics_tracker.bronze.observations.obs_date=source.obs_date').whenNotMatchedInsert(values={'series_id':'source.series_id','obs_date':'source.obs_date','value':'source.value'}).execute()
        ing_log=spark.createDataFrame([{'table':'observations','series_id':s,'error_message':None,'pulled_at':datetime.now(),'rows_inserted':obs_row_count,'status':'Success'}],ing_schema).write.format("delta").mode("append").saveAsTable('us_macroeconomics_tracker.bronze.ingestion_log')
    except Exception as e:
        print(f'Error: {e}')
        ing_log=spark.createDataFrame([{'table':'observations','series_id':s,'error_message':str(e),'pulled_at':datetime.now(),'rows_inserted':0,'status':'Failure'}],ing_schema).write.format("delta").mode("append").saveAsTable('us_macroeconomics_tracker.bronze.ingestion_log')
        

In [0]:
check_schema=StructType([
    StructField('check_name',StringType(),False),
    StructField('table_name',StringType(),False),
    StructField('series_id',StringType(),True),
    StructField('status',StringType(),False),
    StructField('failed_count',LongType(),False),
    StructField('run_at',TimestampType(),False)
])

In [0]:
def run_observations_checks():
    checks=[]
    obs_delta=spark.table('us_macroeconomics_tracker.bronze.observations')
    ing_delta=spark.table('us_macroeconomics_tracker.bronze.ingestion_log')
    null_count=obs_delta.where(F.col('series_id').isNull() | F.col('obs_date').isNull()).count()
    checks.append({
    'check_name': 'null_check',
    'table_name': 'observations',
    'series_id': 'ALL',
    'status': 'PASS' if null_count == 0 else 'FAIL',
    'failed_count': null_count,
    'run_at': datetime.now()
    })
    invalid_series_check=obs_delta.where(~F.col('series_id').isin(*SERIES_ID)).count()
    checks.append({
    'check_name': 'invalid_value_check',
    'table_name': 'observations',
    'series_id': 'ALL',
    'status': 'PASS' if invalid_series_check == 0 else 'FAIL',
    'failed_count': invalid_series_check,
    'run_at': datetime.now()
    })
    duplicate_check=obs_delta.groupBy('series_id', 'obs_date').agg(F.count(F.lit(1)).alias('count')).where(F.col('count') > 1).count()
    checks.append({
    'check_name': 'duplicate_check',
    'table_name': 'observations',
    'series_id': 'ALL',
    'status': 'PASS' if duplicate_check == 0 else 'FAIL',
    'failed_count': duplicate_check,
    'run_at': datetime.now()
    })
    value_check=obs_delta.where((F.col('value')<0) & (F.col('value').isNotNull())).count()
    checks.append({
    'check_name': 'value_check',
    'table_name': 'observations',
    'series_id': 'ALL',
    'status': 'PASS' if value_check == 0 else 'FAIL',
    'failed_count': value_check,
    'run_at': datetime.now()
    })
    w=Window.partitionBy(F.col('series_id')).orderBy(F.col('pulled_at').desc())
    ing_count=ing_delta.where((F.col('table')=='observations') & (F.col('status')=='Success')).withColumn('rn',F.row_number().over(w)).where(F.col('rn')==F.lit(1)).select('series_id','rows_inserted')
    obs_delta_count=obs_delta.groupBy(F.col('series_id')).agg(F.count(F.lit(1)).alias('count'))
    joined_df=ing_count.join(obs_delta_count, on='series_id',how='inner').withColumn('percent_change',((F.col('count') - F.col('rows_inserted')) / F.col('rows_inserted')) * 100).where(F.col('percent_change')< F.lit(-10)).count()    
    checks.append({
    'check_name': 'row_count_check',
    'table_name': 'observations',
    'series_id': 'ALL',
    'status': 'PASS' if joined_df == 0 else 'FAIL',
    'failed_count': joined_df,
    'run_at': datetime.now()
    })
    dq_log=spark.createDataFrame(checks,check_schema).write.format("delta").mode("append").saveAsTable('us_macroeconomics_tracker.bronze.dq_log')

In [0]:
def run_indicators_checks():
    checks=[]
    ind_delta=spark.table('us_macroeconomics_tracker.bronze.indicators')
    completeness_check=ind_delta.select(F.col('series_id')).distinct().count()
    checks.append({
    'check_name': 'completeness_check',
    'table_name': 'indicators',
    'series_id': 'ALL',
    'status': 'PASS' if completeness_check== len(SERIES_ID) else 'FAIL',
    'failed_count': len(SERIES_ID) - completeness_check,
    'run_at': datetime.now()
    })
    null_count=ind_delta.where(F.col('series_name').isNull() | F.col('frequency').isNull() | F.col('units').isNull()).count()
    checks.append({
    'check_name': 'null_check',
    'table_name': 'indicators',
    'series_id': 'ALL',
    'status': 'PASS' if null_count == 0 else 'FAIL',
    'failed_count': null_count,
    'run_at': datetime.now()
    })
    dq_log=spark.createDataFrame(checks,check_schema).write.format("delta").mode("append").saveAsTable('us_macroeconomics_tracker.bronze.dq_log')

In [0]:
%sql
OPTIMIZE us_macroeconomics_tracker.bronze.observations 
ZORDER BY (series_id, obs_date)

In [0]:
%sql
OPTIMIZE us_macroeconomics_tracker.bronze.indicators 
ZORDER BY (series_id)

In [0]:
def send_dq_alert():
    dq_delta=spark.table('us_macroeconomics_tracker.bronze.dq_log')
    latest_run=dq_delta.agg(F.max('run_at')).collect()[0][0]
    failed_count=dq_delta.where((F.col('run_at')==latest_run) & (F.col('status')=='FAIL')).count()
    if failed_count==0:
        print('ALL CHECKS PASSED. NO MAIL SENT.')
        return
    else:
        failures = dq_delta.where((F.col('run_at') == latest_run) & (F.col('status') == 'FAIL')).collect()
        body_lines = []
        for row in failures:
            body_lines.append(f"Check_name:{row['check_name']} - Table_name:{row['table_name']} - Failed_Count:{row['failed_count']} - Run_At:{row['run_at']}")
        body_text = '\n'.join(body_lines)
        message = MIMEMultipart("alternative")
        message["Subject"] = f"DQ Alert — US Macroeconomics Pipeline ran on {latest_run}"
        message["From"] = SENDER_EMAIL
        message["To"] = RECEIVER_EMAIL
        text = f"""\
        DQ ALERT!! PIPELINE FAILED on {latest_run}.

        {body_text}

        TO LEARN MORE, NAVIGATE TO YOUR NOTEBOOK AND QUERY:
        SELECT * FROM us_macroeconomics_tracker.bronze.dq_log WHERE run_at='{latest_run}' and STATUS='FAIL'
        ORDER BY run_at desc;
        """
        html = f"""\
            <html>
                <body>
                    <p>DQ ALERT!! PIPELINE FAILED on {latest_run}.<br>
                       {body_text}</p>
                    <p> TO LEARN MORE, NAVIGATE TO YOUR NOTEBOOK AND QUERY:<br>
                        SELECT * FROM us_macroeconomics_tracker.bronze.dq_log WHERE run_at='{latest_run}' and STATUS='FAIL'
                        ORDER BY run_at desc;</p>
                </body>
            </html>
        """
        part1 = MIMEText(text, "plain")
        part2 = MIMEText(html, "html")
        message.attach(part1)
        message.attach(part2)
        try:
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as SMTP_SERVER:
                SMTP_SERVER.login(SENDER_EMAIL, SENDER_PASSWORD)
                SMTP_SERVER.sendmail(SENDER_EMAIL, RECEIVER_EMAIL, message.as_string())
                print('EMAIL SENT!')
        except Exception as e:
            print(e)

In [0]:
def run_all_checks():
    run_observations_checks()
    run_indicators_checks()
    send_dq_alert()
run_all_checks()

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.bronze.gdp_percent_growth 
USING DELTA
AS
(
  WITH cte1 as(
    SELECT AVG(`value`) AS avg_gdp_value FROM us_macroeconomics_tracker.bronze.observations
    WHERE `series_id`='GDP' AND YEAR(`obs_date`)=2000
  ),
  cte2 AS(
    select `observation_id`,`series_id`,`obs_date`,`value`,(((`value`-(select `avg_gdp_value` from cte1))/(select `avg_gdp_value` from cte1))*100) as percent_growth
    from us_macroeconomics_tracker.bronze.observations
    where `series_id`='GDP'
  )select `observation_id`,`series_id`,`obs_date`,`value`,`percent_growth` from cte2
);

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.bronze.highest_mortage_rate 
USING DELTA
AS
(
  with cte1 as
  (
    select `observation_id`,`series_id`,`obs_date`,`value`, dense_rank() over (partition by year(`obs_date`) order by `value` desc) as rnk 
    from us_macroeconomics_tracker.bronze.observations where `series_id`='MORTGAGE30US'
  )
  select `observation_id`,`series_id`,`obs_date`,`value` from cte1 where `rnk`=1
)

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.bronze.mom_rate_change 
USING DELTA
AS
(
  with cte1 as
  (
    select `observation_id`,`series_id`,`value`,date_format(`obs_date`,'yyyyMM') as `yearmonth`,
    lag(`value`) over (order by `obs_date`) as `Month_Over_Month`
    from us_macroeconomics_tracker.bronze.observations
    where `series_id`='FEDFUNDS'
  ),
  cte2 as 
  (
    select `observation_id`,`series_id`,`value`,`yearmonth`,`Month_Over_Month`,
    (`value`-`Month_Over_Month`) as `difference`,
    case
        when ((`value` - `Month_Over_Month`) > 0) then 'Hike'
        when ((`value` - `Month_Over_Month`) < 0) then 'Cut'
        when ((`value` - `Month_Over_Month`) = 0) then 'No Change'
        else 'No Data'
    end AS `Change_Direction`
    from cte1
  )
  select `observation_id`,`series_id`,`value`,`yearmonth`,`Month_Over_Month`,`difference`,
  `Change_Direction` from cte2
)

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.bronze.rate_hike_cycle 
USING DELTA
AS
(
  WITH cte1 as
  (
    select `observation_id`,`yearmonth`,row_number() over (order by `yearmonth`) as `rn1`
    from us_macroeconomics_tracker.bronze.mom_rate_change 
  ),
  cte2 as
  (
    select `observation_id`,`yearmonth`,row_number() over (order by `yearmonth`) as `rn2`
    from us_macroeconomics_tracker.bronze.mom_rate_change 
    where `Change_Direction`='Hike'
  ),
  cte3 as
  (
    select
    (`c1`.`rn1`-`c2`.`rn2`) as `streak_group`,
    min(`c2`.`yearmonth`) as `cycle_start`,
    max(`c2`.`yearmonth`) as `cycle_end`,
    count(1) `duration`
    from cte1 c1
    join cte2 c2 on c1.observation_id=c2.observation_id
    group by `streak_group`
  )
  select `streak_group`,`cycle_start`,`cycle_end`,`duration` from cte3
)

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.bronze.recession_predictor 
USING DELTA
AS
(
  with dgs10 as
  (
    select date_format(`obs_date`,'yyyyMM') as `yearmonth`,
    avg(`value`) as `avg_dgs10`
    from us_macroeconomics_tracker.bronze.observations 
    where `series_id`='DGS10'
    group by `yearmonth`
  ),
  fedfunds as
  (
    select date_format(`obs_date`,'yyyyMM') as `yearmonth`,
    avg(`value`) as `avg_fedfunds`
    from us_macroeconomics_tracker.bronze.observations 
    where `series_id`='FEDFUNDS'
    group by `yearmonth`
  ),
  cte1 as
  (
    select d.`yearmonth`,
    (d.`avg_dgs10`-f.`avg_fedfunds`) as `spread`,
    case
        when ((d.`avg_dgs10` - f.`avg_fedfunds`) > 0) then 'Positive'
        when ((d.`avg_dgs10` - f.`avg_fedfunds`) < 0) then 'Negative'
        else 'Zero'
    end AS `predictor`
    from dgs10 d
    join fedfunds f on d.`yearmonth`=f.`yearmonth`
  )
  select `yearmonth`,`spread`,`predictor` from cte1
)

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.bronze.rolling_avg_cpi 
USING DELTA
AS
(
  with cte1 as
  (
    select `observation_id`,`series_id`,`obs_date`,`value`,
    row_number() over (order by `obs_date`) as `rn`
    from us_macroeconomics_tracker.bronze.observations
    where `series_id`='CPIAUCSL'
  ),
  cte2 as
  (
    select `observation_id`,`series_id`,`obs_date`,`value`,
    avg(`value`) over (order by `obs_date` rows between 11 preceding and current row) as `AVG_CPI`
    from cte1
    where `rn`>=12
  ),
  cte3 as
  (
    select `observation_id`,`series_id`,`obs_date`,`value`,`AVG_CPI`,
    (`value`-`AVG_CPI`) as `deviation`,
    case
        when ((`value` - `AVG_CPI`) <= 0) then 'Below Trend'
        else 'Above Trend'
    end AS `Trends`
    from cte2
  )
  select `observation_id`,`series_id`,`obs_date`,`value`,`AVG_CPI`,`deviation`,`Trends`
  from cte3
)

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.bronze.year_2_year_change 
USING DELTA
AS
(
  with cte1 as
  (
    select `observation_id`,`series_id`,`obs_date`,`value`,
    `previous`,(((`value`-`previous`)/`previous`)*100) as `year_2_year`
    from
    (
      select `observation_id`,`series_id`,`obs_date`,`value`,
      lag(`value`,12) over (order by `obs_date`) as `previous`
      from us_macroeconomics_tracker.bronze.observations
      where `series_id`='M2SL'
    )c1
  )
  select `observation_id`,`series_id`,`obs_date`,`value`,`previous`,`year_2_year`
  from cte1
)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS us_macroeconomics_tracker.silver;

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.silver.wide_macro_indicators
USING DELTA
AS
with cte1 as
(
select o.obs_date,o.value,o.series_id,i.frequency from us_macroeconomics_tracker.bronze.observations o inner join us_macroeconomics_tracker.bronze.indicators i on o.series_id=i.series_id
),
monthly_quarterly as
(
select obs_date,value,series_id from cte1 
where frequency in ('Quarterly','Monthly')
),
weekly_daily as 
(
select series_id,DATE_TRUNC('month', obs_date) as obs_date,avg(value) as value
from cte1
where frequency LIKE '%Weekly%' or frequency='Daily'
group by DATE_TRUNC('month', obs_date),series_id
),
combined as
(
select obs_date,value,series_id from monthly_quarterly
WHERE obs_date >= '1947-01-01'
union all
select obs_date,value,series_id from weekly_daily
WHERE obs_date >= '1947-01-01'
),
pivoted as 
(
select obs_date,
MAX(case when series_id='CPIAUCSL' then value end) as CPI,
MAX(case when series_id='DGS10' then value end) as DGS10,
MAX(case when series_id='FEDFUNDS' then value end) as FEDFUNDS,
MAX(case when series_id='M2SL' then value end) as M2SL,
MAX(case when series_id='UNRATE' then value end) as UNRATE,
MAX(case when series_id='MORTGAGE30US' then value end) as MORTGAGE30US,
MAX(case when series_id='GDP' then value end) as GDP,
MAX(CASE WHEN series_id='USREC' THEN value END) as is_recession
from combined
group by obs_date
order by obs_date
),
final as
(
select obs_date,CPI,DGS10,FEDFUNDS,M2SL,UNRATE,MORTGAGE30US,GDP,is_recession,
case when month(obs_date) in (1,4,7,10) then 1 else 0 end as is_quarter_start
from pivoted
)select obs_date,CPI,DGS10,FEDFUNDS,M2SL,UNRATE,MORTGAGE30US,GDP,is_quarter_start,is_recession from final

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS us_macroeconomics_tracker.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE us_macroeconomics_tracker.gold.annual_macro_summary
USING DELTA
AS
with cte1 as
(
select year(obs_date) as obs_year,
avg(CPI) as avg_cpi,
avg(DGS10) as avg_dgs10,
avg(FEDFUNDS) as avg_fedfunds,
avg(M2SL) as avg_m2sl,
avg(UNRATE) as avg_unrate,
avg(MORTGAGE30US) as avg_mortgage30us,
avg(GDP) as avg_gdp
FROM us_macroeconomics_tracker.silver.wide_macro_indicators
group by year(obs_date)
),
cte2 as
(
select obs_year,avg_cpi,avg_dgs10,avg_fedfunds,avg_m2sl,avg_unrate,avg_mortgage30us,avg_gdp,
lag(avg_gdp) over (order by obs_year) as prev_gdp,
lag(avg_cpi) over (order by obs_year) as prev_cpi
from cte1
),
final as
(
select obs_year,avg_cpi,avg_dgs10,avg_fedfunds,avg_m2sl,avg_unrate,avg_mortgage30us,avg_gdp,
((avg_gdp-prev_gdp)/prev_gdp)*100 as yoy_gdp_growth,
((avg_cpi-prev_cpi)/prev_cpi)*100 as yoy_cpi_growth
from cte2
)select obs_year,avg_cpi,avg_dgs10,avg_fedfunds,avg_m2sl,avg_unrate,avg_mortgage30us,avg_gdp,yoy_gdp_growth,yoy_cpi_growth from final

In [0]:
print("Pipeline completed successfully")